# Credit Default Models: Threshold Tuning by Business Impact

Comparing two default-prediction models by net business impact (cost of false negatives vs. false positives) across decision thresholds, instead of by a single accuracy figure.

_Datasets are not included in this repository._

In [6]:
import pandas as pd
cv = pd.read_csv('cv_scores.csv')
cv.head(10)

,fold,model_a_f1,model_b_f1
0,1,0.61,0.63
1,2,0.59,0.60
2,3,0.65,0.66
3,4,0.63,0.62
4,5,0.60,0.61
5,6,0.62,0.65
6,7,0.66,0.64
7,8,0.58,0.59
8,9,0.61,0.63
9,10,0.64,0.65


In [7]:
eval = pd.read_csv('predictions_eval.csv')
eval.head(10)

,customer_id,actual_default,proba_model_a,proba_model_b,income_level,region,tenure_months,historical_late_payments
0,CUST_0001,0,0.061985,0.738054,Medium,Jakarta,48,2
1,CUST_0002,1,0.958718,0.520669,Medium,Surabaya,2,3
2,CUST_0003,0,0.535938,0.077115,Low,Luar Jawa,6,0
3,CUST_0004,0,0.403926,0.443493,High,Surabaya,28,1
4,CUST_0005,0,0.467529,0.119346,Medium,Luar Jawa,53,1
5,CUST_0006,0,0.404853,0.208193,Low,Surabaya,29,1
6,CUST_0007,0,0.057556,0.226601,High,Surabaya,26,1
7,CUST_0008,1,0.829577,0.000000,High,Luar Jawa,25,3
8,CUST_0009,0,0.490123,0.216402,High,Jakarta,12,0
9,CUST_0010,0,0.476287,0.040671,Medium,Surabaya,5,0


In [10]:
from sklearn.metrics import confusion_matrix

def business_impact(y_true, y_proba, threshold):
    y_pred = (y_proba >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    nt = tp * 8000000 - fn* 8000000 - fp* 400000

    return {
        'threshold': threshold,
        'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
        'net_impact': nt,
        'cost_FN': fn* 8000000,
        'cost_FP': fp* 400000
    }

In [15]:
thresholds=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
results=[]
for index, threshold in enumerate(thresholds):
  a = business_impact(eval['actual_default'], eval['proba_model_a'], threshold)
  b = business_impact(eval['actual_default'], eval['proba_model_b'], threshold)
  results.append({
      'threshold': threshold,
      'model_a': a['net_impact'],
      'model_b': b['net_impact']
  })
results = pd.DataFrame(results)
results

,threshold,model_a,model_b
0,0.1,548800000,407600000
1,0.2,552400000,384400000
2,0.3,556000000,260800000
3,0.4,450800000,288800000
4,0.5,346000000,242800000
5,0.6,150000000,59200000
6,0.7,-32400000,-30400000
7,0.8,-171200000,-250400000
8,0.9,-296800000,-394000000


In [17]:
highest_a=results.nlargest(1,'model_a')
highest_b=results.nlargest(1,'model_b')
print(highest_a)
print(highest_b)

   threshold    model_a    model_b
2        0.3  556000000  260800000
   threshold    model_a    model_b
0        0.1  548800000  407600000
